In [1]:
!pip install Xtr

In [3]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType

spark = (
    SparkSession.builder
    .appName("binance-etl")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")     # crypto trades in UTC
    .config("spark.driver.memory", "12g")             # avoid OOM on the write
    .config("spark.sql.shuffle.partitions", "50")
    .getOrCreate()
)

**HAR-RV**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

SRC = "../data/klines_processed/features_with_targets"
X_COLS = ["rv_15m", "rv_1h", "rv_24h"]          # short / medium / long horizons
har_cols = X_COLS + ["target_15m", "event_time"]

raw = pd.read_parquet(SRC, columns=har_cols).dropna()
raw = raw.sort_values("event_time")

In [ ]:
t0, t1 = raw.event_time.min(), raw.event_time.max()
span = t1 - t0
train_cut = t0 + span * 0.8
val_cut   = t0 + span * 0.9

train = raw[raw.event_time < train_cut].copy()
val   = raw[(raw.event_time >= train_cut) & (raw.event_time < val_cut)].copy()
test  = raw[raw.event_time >= val_cut].copy()

In [ ]:
for c in X_COLS:
    train[c] = np.log1p(train[c])
    val[c]   = np.log1p(val[c])
    test[c]  = np.log1p(test[c])

# ---- Fit HAR-RV (just a 3-input linear regression) ----
har = LinearRegression().fit(train[X_COLS], train["target_15m"])
print("coefficients:", dict(zip(X_COLS, har.coef_)))
print("intercept:   ", har.intercept_)

In [ ]:
def rmse(a, b):
    return np.sqrt(np.mean((a - b) ** 2))

val_pred = har.predict(val[X_COLS])
baseline = train["target_15m"].mean()

print("HAR val RMSE:     ", rmse(val_pred, val["target_15m"].values))
print("predict-mean RMSE:", rmse(val["target_15m"].values, baseline))

**LSTM**

In [ ]:
import numpy as np
import pandas as pd

base = "../data/scaled"
FEATURES = ["rv_15m", "rv_1h", "rv_4h", "rv_24h",
            "parkinson_15m", "parkinson_1h", "volume_zscore", "buy_ratio"]
TARGET = "target_15m"
SEQ_LEN = 48

def load_5min(split):
    df = pd.read_parquet(f"{base}/{split}",
                         columns=["symbol", "event_time"] + FEATURES + [TARGET])
    df = df.sort_values(["symbol", "event_time"])
    df = df.groupby("symbol", group_keys=False, observed=True).apply(lambda g: g.iloc[::5])
    return df.dropna()

train_df = load_5min("train")
val_df   = load_5min("val")
print(len(train_df), len(val_df))

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_FEATURES = Xtr.shape[2]   # 8
EMB_DIM    = 16
HIDDEN     = 64

class VolLSTM(nn.Module):
    def __init__(self, n_features, n_symbols, emb_dim=16, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(n_symbols, emb_dim)
        self.lstm = nn.LSTM(
            input_size=n_features + emb_dim,
            hidden_size=hidden,
            num_layers=2,
            dropout=0.2,
            batch_first=True,
        )
        self.head = nn.Linear(hidden, 1)

    def forward(self, x, sym_id):
        e = self.emb(sym_id)
        e = e.unsqueeze(1).expand(-1, x.size(1), -1)
        x = torch.cat([x, e], dim=2)
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(last).squeeze(1)

model = VolLSTM(N_FEATURES, n_symbols, EMB_DIM, HIDDEN).to(device)
print("device:", device, "| params:", sum(p.numel() for p in model.parameters()))

In [ ]:
def build_sequences(df):
    Xs, ys, syms = [], [], []
    for sym, g in df.groupby("symbol", observed=True):
        feats = g[FEATURES].to_numpy(np.float32)
        tgt   = g[TARGET].to_numpy(np.float32)
        if len(g) <= SEQ_LEN:
            continue
        # sliding windows, vectorized — no Python loop over rows
        w = np.lib.stride_tricks.sliding_window_view(feats, SEQ_LEN, axis=0)
        w = w.transpose(0, 2, 1)          # -> (n_windows, SEQ_LEN, n_features)
        Xs.append(w[:-1])                 # drop last: its target is out of range
        ys.append(tgt[SEQ_LEN:])          # target = step AFTER each window
        syms.append(np.full(len(w)-1, sym))
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(syms)

Xtr, ytr, symtr = build_sequences(train_df)
Xval, yval, symval = build_sequences(val_df)
print("Xtr:", Xtr.shape, "ytr:", ytr.shape)
print("Xval:", Xval.shape)

In [ ]:
import numpy as np

symbols = sorted(np.unique(symtr))
sym2id = {s: i for i, s in enumerate(symbols)}
n_symbols = len(symbols)

idtr  = np.array([sym2id[s] for s in symtr],  dtype=np.int64)
idval = np.array([sym2id[s] for s in symval], dtype=np.int64)

print("n_symbols:", n_symbols)        # expect 20
print("idtr:", idtr.shape, "range:", idtr.min(), "-", idtr.max())

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def make_loader(X, y, ids, batch=512, shuffle=False):
    ds = TensorDataset(
        torch.from_numpy(X),                  # float32 already
        torch.from_numpy(ids),                # int64
        torch.from_numpy(y),                  # float32
    )
    return DataLoader(ds, batch_size=batch, shuffle=shuffle)

train_loader = make_loader(Xtr, ytr, idtr, batch=512, shuffle=True)
val_loader   = make_loader(Xval, yval, idval, batch=512, shuffle=False)

print("train batches:", len(train_loader), "| val batches:", len(val_loader))

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_FEATURES = Xtr.shape[2]
EMB_DIM, HIDDEN = 16, 64

class VolLSTM(nn.Module):
    def __init__(self, n_features, n_symbols, emb_dim=16, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(n_symbols, emb_dim)
        self.lstm = nn.LSTM(n_features + emb_dim, hidden,
                            num_layers=2, dropout=0.2, batch_first=True)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x, sym_id):
        e = self.emb(sym_id).unsqueeze(1).expand(-1, x.size(1), -1)
        out, _ = self.lstm(torch.cat([x, e], dim=2))
        return self.head(out[:, -1, :]).squeeze(1)

model = VolLSTM(N_FEATURES, n_symbols, EMB_DIM, HIDDEN).to(device)
print("device:", device)

In [ ]:
import numpy as np
y_mean, y_std = ytr.mean(), ytr.std()
print("y_mean:", y_mean, "y_std:", y_std)

ytr_s  = ((ytr  - y_mean) / y_std).astype(np.float32)
yval_s = ((yval - y_mean) / y_std).astype(np.float32)

train_loader = make_loader(Xtr, ytr_s, idtr, batch=512, shuffle=True)
val_loader   = make_loader(Xval, yval_s, idval, batch=512, shuffle=False)

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()   # matches the target: MSE on log-variance

EPOCHS = 8

for epoch in range(EPOCHS):
    # ---- train ----
    model.train()
    tr_loss = 0.0
    for xb, idb, yb in train_loader:
        xb, idb, yb = xb.to(device), idb.to(device), yb.to(device)
        opt.zero_grad()
        pred = model(xb, idb)
        loss = loss_fn(pred, yb)
        loss.backward()
        opt.step()
        tr_loss += loss.item() * len(yb)
    tr_loss /= len(train_loader.dataset)

    # ---- validate ----
    model.eval()
    va_loss = 0.0
    with torch.no_grad():
        for xb, idb, yb in val_loader:
            xb, idb, yb = xb.to(device), idb.to(device), yb.to(device)
            pred = model(xb, idb)
            va_loss += loss_fn(pred, yb).item() * len(yb)
    va_loss /= len(val_loader.dataset)

    print(f"epoch {epoch+1}: train MSE {tr_loss:.3e} | val MSE {va_loss:.3e}")

In [ ]:
import numpy as np
model.eval(); preds, trues = [], []
with torch.no_grad():
    for xb, idb, yb in val_loader:
        xb, idb = xb.to(device), idb.to(device)
        preds.append(model(xb, idb).cpu().numpy()); trues.append(yb.numpy())
preds_o = np.concatenate(preds) * y_std + y_mean
trues_o = np.concatenate(trues) * y_std + y_mean
print("LSTM val RMSE:", np.sqrt(np.mean((preds_o - trues_o)**2)))
print("mean val RMSE:", np.sqrt(np.mean((trues_o - y_mean)**2)))
print("HAR  val RMSE: 2.005e-05")

In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "y_mean": float(y_mean), "y_std": float(y_std),
    "sym2id": sym2id,
    "config": {"n_features": N_FEATURES, "n_symbols": n_symbols,
               "emb_dim": EMB_DIM, "hidden": HIDDEN, "seq_len": SEQ_LEN},
}, "../data/lstm_model.pt")
print("saved")